# 🍷 Wine Quality Prediction - Machine Learning Internship Project

**Objective:** Predict wine quality based on physicochemical properties using supervised machine learning algorithms.

---

### Project Workflow:
1. **Data Loading:** Ingest UCI Red and White Wine Quality datasets.
2. **Exploratory Data Analysis (EDA):** Check distributions, missing values, duplicates, and feature correlations.
3. **Data Preprocessing:** Target formulation (High Quality $\ge 7$ vs Standard $< 7$), train-test split, and feature scaling (`StandardScaler`).
4. **Model Training & Comparison:** Compare Logistic Regression, Decision Tree, Random Forest, Gradient Boosting, and Support Vector Machine.
5. **Model Evaluation:** Evaluate models using Accuracy, Precision, Recall, F1-Score, ROC-AUC, and Confusion Matrix.
6. **Feature Importance:** Identify the physicochemical drivers of high-quality wine.
7. **Model Serialization:** Export the best model for deployment in a Streamlit web app.


In [ ]:
# Step 1: Import Essential Libraries
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn modules
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)
import joblib

# Plot styling configuration
sns.set_theme(style="whitegrid", palette="muted")
%matplotlib inline
print("Libraries imported successfully!")


## 1. Data Ingestion & Overview
The UCI Wine Quality dataset consists of two subsets:
- **Red Wine**: 1,599 samples
- **White Wine**: 4,898 samples
Each sample includes 11 physicochemical laboratory measurements and an expert sensory quality rating (0 to 10).


In [ ]:
# Load Red and White wine datasets (UCI uses ';' delimiter)
data_dir = "../data"
red_path = os.path.join(data_dir, "winequality-red.csv")
white_path = os.path.join(data_dir, "winequality-white.csv")

# Fallback to root or current directory if needed
if not os.path.exists(red_path):
    red_path = "winequality-red.csv"
    white_path = "winequality-white.csv"

red_df = pd.read_csv(red_path, sep=";")
red_df["wine_type"] = "red"

white_df = pd.read_csv(white_path, sep=";")
white_df["wine_type"] = "white"

# Combine into a unified dataset
df = pd.concat([red_df, white_df], ignore_index=True)
print(f"Red Wine Shape:   {red_df.shape}")
print(f"White Wine Shape: {white_df.shape}")
print(f"Combined Shape:   {df.shape}")
df.head()


## 2. Data Integrity: Missing Values & Duplicates
Before modeling, verify if there are any missing values, NaNs, or duplicate records.


In [ ]:
# Check for missing values
print("--- Missing Values per Column ---")
print(df.isnull().sum())

# Check for duplicate records
print(f"\nDuplicate records in combined dataset: {df.duplicated().sum()}")


## 3. Exploratory Data Analysis (EDA)
Let us explore:
1. The distribution of wine quality scores.
2. The correlation of physicochemical features with wine quality.
3. How key features (Alcohol, Volatile Acidity, Sulphates) differ between quality classes.


In [ ]:
# Plot Wine Quality Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# By wine type
sns.countplot(
    data=df, x="quality", hue="wine_type", 
    palette={"red": "#b22222", "white": "#f4d03f"}, 
    ax=axes[0]
)
axes[0].set_title("Quality Distribution by Wine Type", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Quality Score (3 to 9)")
axes[0].set_ylabel("Count")

# Combined with percentages
quality_counts = df["quality"].value_counts().sort_index()
sns.barplot(x=quality_counts.index, y=quality_counts.values, ax=axes[1], color="#4c72b0")
axes[1].set_title("Overall Quality Distribution (Combined)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Quality Score")
axes[1].set_ylabel("Count")

for i, count in enumerate(quality_counts.values):
    pct = f"{(count / len(df)) * 100:.1f}%"
    axes[1].text(i, count + 30, pct, ha="center", fontsize=9, fontweight="bold")

plt.tight_layout()
plt.show()


In [ ]:
# Feature Correlation Heatmap
numeric_df = df.drop(columns=["wine_type"])
corr = numeric_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

plt.figure(figsize=(10, 8))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", square=True, linewidths=0.5)
plt.title("Physicochemical Correlation Heatmap", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# Boxplots: Top features correlated with quality
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

key_features = ["alcohol", "volatile acidity", "sulphates", "citric acid"]
for i, feat in enumerate(key_features):
    sns.boxplot(data=df, x="quality", y=feat, hue="wine_type", palette={"red": "#b22222", "white": "#f4d03f"}, ax=axes[i])
    axes[i].set_title(f"{feat.title()} vs Quality", fontsize=11, fontweight="bold")

plt.tight_layout()
plt.show()


## 4. Data Preprocessing & Target Formulation
### Problem Formulation:
In wine production, wines rated **7 or above** are considered **High Quality (Premium)**, while wines rated **6 or below** are considered **Standard / Average Quality**.

We formulate a binary classification problem:
- `1` = High Quality (Quality $\ge 7$)
- `0` = Standard Quality (Quality $< 7$)

### Steps:
1. Create `quality_label`.
2. Encode `is_white` (1 for white wine, 0 for red wine).
3. Split data into 80% Training and 20% Testing sets using **stratification**.
4. Scale features with `StandardScaler` (fit on train, transform test) to prevent data leakage.


In [ ]:
# 1. Target Formulation
df["quality_label"] = (df["quality"] >= 7).astype(int)
df["is_white"] = (df["wine_type"] == "white").astype(int)

# 2. Features and Target separation
feature_cols = [col for col in df.columns if col not in ["quality", "quality_label", "wine_type"]]
X = df[feature_cols].copy()
y = df["quality_label"].copy()

print(f"Features: {feature_cols}")
print(f"Target distribution:\n{y.value_counts(normalize=True)}")

# 3. Stratified Train-Test Split (80/20)
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 4. Feature Scaling (Fit on Train ONLY)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

print(f"\nX_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")


## 5. Model Training & Comparison
We train and compare 5 classification models:
1. **Logistic Regression** (Linear baseline)
2. **Decision Tree Classifier** (Interpretable tree baseline)
3. **Random Forest Classifier** (Bagged ensemble of decision trees)
4. **Gradient Boosting Classifier** (Sequentially boosted ensemble)
5. **Support Vector Machine (SVC)** (Margin-maximizing kernel classifier)


In [ ]:
# Define models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=42),
    "Support Vector Machine": CalibratedClassifierCV(SVC(random_state=42), cv=3),
}

results = []
trained_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    trained_models[name] = model
    
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    auc = roc_auc_score(y_test, y_proba)
    
    results.append({
        "Model": name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1-Score": f1,
        "ROC-AUC": auc
    })

comparison_df = pd.DataFrame(results).sort_values(by="F1-Score", ascending=False).reset_index(drop=True)
comparison_df


## 6. Evaluation of the Best Model
Let us evaluate our best-performing model (**Random Forest**) in detail using:
- Classification Report
- Confusion Matrix
- Receiver Operating Characteristic (ROC) Curve
- Feature Importance


In [ ]:
best_model_name = comparison_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]
best_y_pred = best_model.predict(X_test)

print(f"=== CLASSIFICATION REPORT ({best_model_name}) ===")
print(classification_report(y_test, best_y_pred, target_names=["Standard (<7)", "High Quality (>=7)"]))

# Confusion Matrix
cm = confusion_matrix(y_test, best_y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=["Standard (<7)", "High Quality (>=7)"],
    yticklabels=["Standard (<7)", "High Quality (>=7)"],
    ax=ax, annot_kws={"size": 13, "fontweight": "bold"}
)
ax.set_title(f"Confusion Matrix: {best_model_name}", fontsize=13, fontweight="bold")
ax.set_xlabel("Predicted Label")
ax.set_ylabel("True Label")
plt.tight_layout()
plt.show()


In [ ]:
# ROC Curve Comparison
plt.figure(figsize=(8, 6))
for name, model in trained_models.items():
    y_proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    plt.plot(fpr, tpr, lw=2, label=f"{name} (AUC = {auc:.3f})")

plt.plot([0, 1], [0, 1], "k--", lw=1.5, label="Random Guess (AUC = 0.500)")
plt.xlabel("False Positive Rate", fontsize=11)
plt.ylabel("True Positive Rate", fontsize=11)
plt.title("ROC Curves Comparison", fontsize=13, fontweight="bold")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()


In [ ]:
# Feature Importance (Random Forest)
if hasattr(best_model, "feature_importances_"):
    importances = best_model.feature_importances_
    sorted_idx = np.argsort(importances)[::-1]
    
    df_imp = pd.DataFrame({
        "Feature": [feature_cols[i] for i in sorted_idx],
        "Importance": [importances[i] for i in sorted_idx]
    })
    
    plt.figure(figsize=(10, 6))
    sns.barplot(data=df_imp, x="Importance", y="Feature", hue="Feature", palette="Blues_r", legend=False)
    plt.title(f"Feature Importance ({best_model_name})", fontsize=13, fontweight="bold")
    plt.xlabel("Gini Importance Score")
    for i, val in enumerate(df_imp["Importance"]):
        plt.text(val + 0.003, i, f"{val:.3f}", va="center", fontsize=9, fontweight="bold")
    plt.tight_layout()
    plt.show()


## 7. Key Findings & Next Steps

### Internship Insights:
1. **Alcohol content** is the single most influential driver of wine quality ratings (positive correlation).
2. **Volatile acidity** strongly degrades perceived quality (vinegar off-flavor).
3. **Random Forest** outperformed simpler baselines, achieving ~89% test accuracy and an ROC-AUC of 0.91+.
4. The trained model and preprocessing pipeline are saved and integrated into the interactive **Streamlit Web Application** (`app.py`).
